# Three-operand operator controls · Qwen3.5-4B

**Same experiments as the minimal Qwen3-VL-4B notebook**, using the same 48 saved CLEVR images, three rule expressions, sampling seeds, image-set coverage, prompt wording, 72-trial design, controls, scoring metrics, and ablation/perception/gate diagnostics. No image regeneration.

**Fresh Colab GPU session:** run **Setup** cells, then all **Definitions** cells, then **Readout sanity** and **★ Main 72 trials**. The optional controls and diagnostics run only if you uncomment their calls. Every saved run resumes from Drive without repeating completed calls.

**Model:** `Qwen/Qwen3.5-4B`, 4-bit NF4, native Hugging Face adapter because Roshan's Qwen3-VL wrapper is not known to support Qwen3.5. **Thinking mode disabled** for short-label scoring; X/Y/Z candidate probabilities are computed from the **next-token logits**, normalized across those three labels, analogous to Roshan's readout. The adapter differs from Roshan's model wrapper, so compare behavioral results with appropriate caution.

**Drive outputs:** `MyDrive/LogicCBM++/three_operand_exp8_qwen35_minimal/` — the original Qwen3-VL results are untouched. **Runtime:** a CUDA GPU (T4 for 4-bit if enough memory). Run the label sanity checks before using the main results.

**Limitations:** The model has not been executed in Colab here. Passing ten images may be GPU-memory intensive. The one-token logits method may reflect label preference instead of rule inference; position/relabel and text-only controls are essential.


## Setup · run after every runtime restart


In [ ]:
# Install on every fresh Colab runtime. Restart the runtime if Colab asks you to.
%pip install -q -U "transformers>=5.0" accelerate bitsandbytes pillow scikit-learn

import torch, transformers
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__,
      "| transformers:", transformers.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
from collections import Counter, defaultdict
from itertools import product, permutations
import random, json, zipfile, shutil, sys, math, re, hashlib
import numpy as np

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/LogicCBM++/three-operand-experiments')
DATA_DRIVE = ROOT / 'data'
DATA = Path('/content/three_operand_pilot')
OUT = ROOT / 'results' / 'qwen3.5-4b'
OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.5-4B'

# Reuse the same existing scenes (no fresh generation).
assert (DATA_DRIVE/'metadata.json').is_file(), 'Missing three_operand_pilot/metadata.json in Drive'
shutil.copytree(DATA_DRIVE, DATA, dirs_exist_ok=True)
IMAGE_DIR = DATA/'images'
metadata_path = DATA/'metadata.json'
metadata = json.loads(metadata_path.read_text())
records = metadata['records']
assert len(records) == 48 and all((IMAGE_DIR/r['filename']).is_file() for r in records)
DATA_HASH = hashlib.sha256(metadata_path.read_bytes()).hexdigest()[:16]
print('Existing images:', len(records), '| results:', OUT)


In [ ]:
# Qwen3.5-native multimodal adapter replacing Roshan's Qwen3-VL-specific loader.
# Everything after this cell continues using vlm.chat(...), option_probs(...), TOKEN_IDS.
from PIL import Image
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration, BitsAndBytesConfig
import torch, gc

gc.collect()
torch.cuda.empty_cache()

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4-compatible
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quant,
    dtype=torch.float16,
    attn_implementation='sdpa',
).eval()

LABELS = ('X', 'Y', 'Z')
TOKEN_IDS = {}
for label in LABELS:
    ids = processor.tokenizer.encode(label, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f'{label!r} must be exactly one token: {ids}')
    TOKEN_IDS[label] = ids[0]
assert len(set(TOKEN_IDS.values())) == 3

MAX_PIXELS = 384 * 28 * 28  # Match the previous experiment's per-image limit.

class Qwen35Adapter:
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor

    def _inputs(self, paths, prompt):
        # Match Roshan's input ordering: image(s) followed by question text.
        content = []
        for path in paths:
            with Image.open(path) as f:
                im = f.convert('RGB')
            if im.width * im.height > MAX_PIXELS:
                scale = (MAX_PIXELS / (im.width * im.height)) ** 0.5
                im = im.resize((max(1,int(im.width*scale)),
                                max(1,int(im.height*scale))), Image.Resampling.LANCZOS)
            content.append({'type': 'image', 'image': im})
        content.append({'type': 'text', 'text': prompt})
        messages = [{'role': 'user', 'content': content}]
        return self.processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            enable_thinking=False,  # Important: next token should be the answer.
            return_dict=True, return_tensors='pt',
        ).to(self.model.device)

    @torch.inference_mode()
    def option_probs(self, paths, prompt, token_ids):
        inputs = self._inputs(paths, prompt)
        # Score the *first answer token*, before any text is generated.
        output = self.model(**inputs, logits_to_keep=1, use_cache=False)
        logits = output.logits[0, -1].float()
        values = torch.stack([logits[token_ids[l]] for l in LABELS])
        probabilities = torch.softmax(values, dim=0).cpu().tolist()
        del inputs, output, logits, values
        return dict(zip(LABELS, probabilities))

    @torch.inference_mode()
    def chat(self, paths, prompt, max_new_tokens=150, temperature=0.0):
        inputs = self._inputs(paths, prompt)
        input_len = inputs['input_ids'].shape[1]
        outputs = self.model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic like the prior temperature=0 runs.
            pad_token_id=self.processor.tokenizer.eos_token_id,
        )
        text = self.processor.decode(
            outputs[0, input_len:], skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        del inputs, outputs
        return text

    def info(self):
        return {'model': MODEL_ID, 'quantization':'NF4 4-bit',
                'thinking':False, 'readout':'next token restricted X/Y/Z'}

vlm = Qwen35Adapter(model, processor)

def option_probs(vlm, paths, prompt, token_ids):
    """Same call signature as Roshan's readout; native Qwen3.5 implementation."""
    return vlm.option_probs(paths, prompt, token_ids)

print('Model:', vlm.info())
print('Candidate label token IDs:', TOKEN_IDS)


## Definitions · run after restart, no model calls

In [ ]:
EXPRESSIONS = {'R1': '(A AND B) OR C', 'R2': '(A AND C) OR B', 'R3': '(B AND C) OR A'}
RULES = tuple(EXPRESSIONS)
POOL = [r for r in records if r['variation'] < 4]  # hold out variations 4–5
N_IMAGES, SETS_PER_RULE, SEED = 10, 8, 2026

def satisfies(rule, bits):
    a,b,c = map(bool,bits)
    return {'R1': (a and b) or c, 'R2': (a and c) or b,
            'R3': (b and c) or a}[rule]

def compatible_rules(items):
    return [r for r in RULES if all(satisfies(r, x['bits']) for x in items)]

def sample_set(rule, idx, mode='coverage', n=N_IMAGES, seed=SEED):
    """coverage: all 5 positive types; iid: positive draws with replacement;
       decisive_once: only one one-hot positive scene (for ablation)."""
    assert mode in ('coverage', 'iid', 'decisive_once')
    rng = random.Random(seed + {'coverage':0,'iid':100000,'decisive_once':200000}[mode]
                        + 1000*RULES.index(rule) + idx)
    admissible = [b for b in product((0,1), repeat=3) if satisfies(rule,b)]
    eligible = [r for r in POOL if satisfies(rule,r['bits'])]
    if mode == 'iid':
        chosen = [rng.choice(eligible) for _ in range(n)]
    else:
        assert n >= len(admissible)
        chosen = [rng.choice([r for r in eligible if tuple(r['bits']) == b])
                  for b in admissible]
        unique = next(b for b in admissible if sum(b) == 1)
        remainder = [r for r in eligible if r['filename'] not in {x['filename'] for x in chosen}
                     and (mode != 'decisive_once' or tuple(r['bits']) != unique)]
        chosen += rng.sample(remainder, n-len(chosen))
    rng.shuffle(chosen)
    consistent = compatible_rules(chosen)
    if mode != 'iid':
        assert consistent == [rule]
    if mode == 'decisive_once':
        assert sum(sum(x['bits'])==1 for x in chosen) == 1
    return dict(group_id=f'{mode}:{rule}:{idx}', hidden_rule=rule, set_index=idx,
                sampling=mode, items=chosen, scene_bits=[x['bits'] for x in chosen],
                image_files=[x['filename'] for x in chosen],
                distinct_types_shown=len({tuple(x['bits']) for x in chosen}),
                admissible_types=len(admissible), consistent_candidates=consistent,
                identifiable=(consistent==[rule]))

def layouts(control='cyclic'):
    fixed = dict(zip(LABELS,RULES))
    if control == 'fixed': return [(fixed, LABELS)]
    if control == 'cyclic':
        return [(dict(zip(LABELS,p)), LABELS) for p in
                [('R1','R2','R3'),('R2','R3','R1'),('R3','R1','R2')]]
    if control == 'position':  # fixed label -> rule; change physical line order
        return [(fixed,p) for p in permutations(LABELS)]
    if control == 'relabel':   # fixed expression positions; change X/Y/Z assignments
        return [(dict(zip(p,RULES)),p) for p in permutations(LABELS)]
    raise ValueError(control)

for r in RULES:
    assert sample_set(r,0)['identifiable']
print('Rules and samplers validated:', EXPRESSIONS)

In [ ]:
def scene_text(scene):
    return '\n'.join(f"Scene {i}: A={b[0]}, B={b[1]}, C={b[2]}"
                     for i,b in enumerate(scene['scene_bits'],1))

def build_input(scene, mapping, display, modality='images'):
    """Main image prompt matches the earlier notebook's 72-trial wording."""
    assert modality in ('images','text','captioned')
    choices = '\n'.join(f'{label}: {EXPRESSIONS[mapping[label]]}' for label in display)
    prompt = f"""Images 1-{len(scene["image_files"])} all satisfy ONE hidden Boolean rule.

A is true when a green cube is present.
B is true when a blue sphere is present.
C is true when a red cylinder is present.
Each concept requires the exact colour AND shape.
Any other object is an irrelevant distractor.

Candidate rules:
{choices}

AND requires both inputs to be true.
OR requires at least one input to be true, including both.
Follow the parentheses.

Pick the rule consistent with every image.
Reply with nothing but its label: X, Y, or Z.
ANSWER:"""
    if modality != 'images':
        prompt = ('Ground-truth Boolean scene descriptions (1 = present, 0 = absent):\n'
                  + scene_text(scene) + '\n\n' + prompt.replace('Images 1-', 'Scenes 1-'))
    paths = ([] if modality == 'text' else
             [str(IMAGE_DIR/name) for name in scene['image_files']])
    return paths, prompt

def score_trial(scene, mapping, display, modality='images'):
    paths,prompt = build_input(scene,mapping,display,modality)
    raw = option_probs(vlm,paths,prompt,TOKEN_IDS)  # original Roshan readout
    probs = {label:float(raw[label]) for label in LABELS}
    s = sum(probs.values())
    assert s > 0 and all(math.isfinite(x) and x>=0 for x in probs.values()), probs
    probs = {k:v/s for k,v in probs.items()}
    selected = max(LABELS,key=lambda l:probs[l])
    true_label = next(l for l in LABELS if mapping[l]==scene['hidden_rule'])
    sorted_p = sorted(probs.values(),reverse=True)
    return dict(prompt=prompt, label_mapping=dict(mapping), display_order=list(display),
                modality=modality, probabilities=probs, chosen_label=selected,
                chosen_rule=mapping[selected], true_label=true_label,
                chosen_position=list(display).index(selected)+1,
                correct=(mapping[selected]==scene['hidden_rule']),
                true_probability=probs[true_label], confidence=probs[selected],
                margin=sorted_p[0]-sorted_p[1],
                nll=-math.log(max(probs[true_label],1e-12)),
                brier=sum((probs[l]-float(l==true_label))**2 for l in LABELS),
                trial_entropy_bits=-sum(p*math.log2(p) for p in probs.values() if p>0))

In [ ]:
def entropy(values):
    ct=Counter(values); n=len(values)
    return -sum((k/n)*math.log2(k/n) for k in ct.values() if k)

def mutual_information(truth,pred):
    n=len(truth); a=Counter(truth); b=Counter(pred); ab=Counter(zip(truth,pred))
    return sum((count/n)*math.log2((count*n)/(a[x]*b[y]))
               for (x,y),count in ab.items())

def analyse(rows, run_dir=None, permutations_n=1000, bootstraps=1000):
    assert rows
    truth=[r['hidden_rule'] for r in rows]
    pred=[r['chosen_rule'] for r in rows]
    groups=defaultdict(list)
    for r in rows: groups[r['group_id']].append(r)
    keys=sorted(groups)
    rng=np.random.default_rng(2026)
    observed_mi=mutual_information(truth,pred)
    null=[]
    group_truth=[groups[k][0]['hidden_rule'] for k in keys]
    for _ in range(permutations_n):
        labels=dict(zip(keys,rng.permutation(group_truth)))
        null.append(mutual_information([labels[r['group_id']] for r in rows],pred))
    boot=[]
    for _ in range(bootstraps):
        sampled=rng.choice(keys,size=len(keys),replace=True)
        sample=[r for k in sampled for r in groups[k]]
        boot.append(np.mean([r['correct'] for r in sample]))
    identifiable=[r for r in rows if r['identifiable']]
    summary=dict(trials=len(rows), independent_scene_sets=len(keys),
                 accuracy=float(np.mean([r['correct'] for r in rows])),
                 accuracy_95ci_cluster=[float(x) for x in np.percentile(boot,[2.5,97.5])],
                 identifiable_fraction=len(identifiable)/len(rows),
                 identifiable_accuracy=(float(np.mean([r['correct'] for r in identifiable]))
                                        if identifiable else None),
                 chance_baseline=1/len(RULES),
                 output_entropy_bits=entropy(pred),
                 mutual_information_bits=observed_mi,
                 mi_null_mean=float(np.mean(null)),
                 mi_minus_null=float(observed_mi-np.mean(null)),
                 mi_permutation_p=(1+sum(x>=observed_mi for x in null))/(1+len(null)),
                 mean_true_prob=float(np.mean([r['true_probability'] for r in rows])),
                 mean_nll=float(np.mean([r['nll'] for r in rows])),
                 mean_brier=float(np.mean([r['brier'] for r in rows])),
                 chosen_labels=dict(Counter(r['chosen_label'] for r in rows)),
                 chosen_positions=dict(Counter(str(r['chosen_position']) for r in rows)),
                 chosen_rules=dict(Counter(pred)),
                 per_rule={r:dict(n=sum(x['hidden_rule']==r for x in rows),
                                  accuracy=float(np.mean([x['correct'] for x in rows if x['hidden_rule']==r])))
                           for r in RULES},
                 confusion={r:{p:sum(x['chosen_rule']==p for x in rows if x['hidden_rule']==r)
                                for p in RULES} for r in RULES},
                 by_diversity={str(d):{'n':sum(x['distinct_types_shown']==d for x in rows),
                                       'accuracy':float(np.mean([x['correct'] for x in rows
                                                                 if x['distinct_types_shown']==d]))}
                               for d in sorted({x['distinct_types_shown'] for x in rows})})
    if run_dir is not None:
        (Path(run_dir)/'summary.json').write_text(json.dumps(summary,indent=2))
    print(f"Accuracy: {summary['accuracy']:.1%} | 95% set-bootstrap CI: {summary['accuracy_95ci_cluster']}")
    print(f"MI: {observed_mi:.3f} bits | null mean: {summary['mi_null_mean']:.3f} | p: {summary['mi_permutation_p']:.4f}")
    print('Confusion matrix:',summary['confusion'])
    print('Labels:',summary['chosen_labels'],'| Positions:',summary['chosen_positions'])
    return summary

In [ ]:
def trial_status_line(run_name, completed, total, row):
    """One readable progress line; original trial IDs remain in runs.jsonl."""
    names = {
        'main_72': 'Main experiment', 'text_only': 'Text-only',
        'captioned': 'Captioned', 'position_order': 'Option order',
        'label_assignment': 'Label assignment', 'iid_positive': 'IID sampling',
        'ablation': 'Ablation', 'perception': 'Object recognition',
        'gates': 'Boolean gates',
    }
    name = names.get(run_name, run_name.replace('_', ' ').capitalize())
    prefix = f'{name} {completed}/{total}'
    actual = row.get('hidden_rule')
    predicted = row.get('chosen_rule', row.get('predicted_rule'))
    if actual and predicted is not None:
        verdict = 'Correct' if row.get('correct') else 'Incorrect'
        choice = (f" | Choice {row['chosen_label']} (position {row['chosen_position']})"
                  if 'chosen_label' in row and 'chosen_position' in row else '')
        return f'{prefix} | Actual: {actual} | Predicted: {predicted} | {verdict}{choice}'
    if run_name == 'ablation':
        verdict = 'Correct' if row.get('full_correct') else 'Incorrect'
        return f"{prefix} | Actual: {actual} | Full set: {verdict} | Decisive drop: {row['decisive_drop']:+.3f}"
    if run_name == 'perception':
        return f"{prefix} | {row['filename']} | {'Correct' if row['correct'] else 'Incorrect'}"
    if run_name == 'gates':
        d = 'correct' if row['d_correct'] else 'incorrect'
        y = 'correct' if row['y_correct'] else 'incorrect'
        return f'{prefix} | Rule: {actual} | Intermediate: {d} | Output: {y}'
    return f'{prefix} | Saved'

def saved_runner(run_name, config, jobs, infer):
    """Generic resumable JSONL runner. Exactly one append per completed inference unit."""
    folder=OUT/run_name; folder.mkdir(parents=True,exist_ok=True)
    config_path=folder/'config.json'; rows_path=folder/'runs.jsonl'
    if config_path.exists():
        assert json.loads(config_path.read_text()) == config, f'Config changed: use another run_name: {run_name}'
    else:
        config_path.write_text(json.dumps(config,indent=2))
    rows=[json.loads(x) for x in rows_path.read_text().splitlines() if x.strip()] if rows_path.exists() else []
    done=[r['trial_id'] for r in rows]
    assert len(done)==len(set(done)), f'Duplicate trial ID in {rows_path}'
    seen=set(done)
    for trial_id, payload in jobs:
        if trial_id in seen: continue
        result=infer(payload)
        row={'trial_id':trial_id, **result}
        with rows_path.open('a') as f:
            f.write(json.dumps(row)+'\n')
        rows.append(row); seen.add(trial_id)
        print(trial_status_line(run_name, len(rows), len(jobs), row), flush=True)
    print(f'Completed {len(rows)}/{len(jobs)} trials | Results: {rows_path.parent}')
    return rows,folder

def run_experiment(name='main_72', sampling='coverage', control='cyclic',
                   modality='images', n_sets=SETS_PER_RULE, n_images=N_IMAGES, seed=SEED):
    plan=list(layouts(control))
    config={'version':'exp8-three-operand-v1-qwen35-adapter','model':MODEL_ID,'data_hash':DATA_HASH,
            'sampling':sampling,'control':control,'modality':modality,'sets_per_rule':n_sets,
            'images_per_set':n_images,'seed':seed,'readout':'native Qwen3.5 next-token softmax restricted X/Y/Z', 'thinking':False,
            'expressions':EXPRESSIONS,'labels':list(LABELS),'token_ids':dict(TOKEN_IDS)}
    jobs=[]
    for rule in RULES:
        for i in range(n_sets):
            scene=sample_set(rule,i,sampling,n_images,seed)
            for k,(mapping,display) in enumerate(plan):
                jobs.append((f'{scene["group_id"]}:{control}:{k}:{modality}',
                             (scene,mapping,display)))
    def infer(payload):
        scene,mapping,display=payload
        return dict(group_id=scene['group_id'], hidden_rule=scene['hidden_rule'],
                    set_index=scene['set_index'], sampling=scene['sampling'],
                    image_files=scene['image_files'], scene_bits=scene['scene_bits'],
                    distinct_types_shown=scene['distinct_types_shown'],
                    admissible_types=scene['admissible_types'],
                    identifiable=scene['identifiable'],
                    consistent_candidates=scene['consistent_candidates'],
                    **score_trial(scene,mapping,display,modality))
    rows,folder=saved_runner(name,config,jobs,infer)
    return rows,analyse(rows,folder),folder

## ★ Main experiment · same 72 trials

**3 generating rules × 8 sets of 10 images × 3 cyclic label mappings = 72 calls.** Same random seed and scene generator as the minimal Qwen3-VL notebook. The run records probabilities, confidence, NLL, Brier scores, entropy, accuracy, confusion counts, mutual information, shuffled-MI null, and set-level bootstrap uncertainty.

First run the label-sanity cell below. If the candidate probabilities keep selecting the same letter despite explicitly requesting each different letter, inspect the readout before interpreting model behavior as rule reasoning. The position, relabel and text controls appear below and are optional.


In [ ]:
# ★ Mandatory label-readout sanity + a single image-trial smoke test.
# Check that the candidate scoring can distinguish deliberately requested labels.
for expected in LABELS:
    instruction = (f"This is a format-only test. The correct answer is {expected}. "
                   "Reply with only that single capital letter. ANSWER:")
    p = option_probs(vlm, [], instruction, TOKEN_IDS)
    print('Requested:', expected, '| predicted:', max(p,key=p.get),
          '| restricted P(X,Y,Z):', {k:round(v,5) for k,v in p.items()})

scene = sample_set('R1', 0)
mapping, display = layouts('fixed')[0]
probe = score_trial(scene, mapping, display, 'images')
print('Image smoke test: expected R1 | predicted:', probe['chosen_rule'],
      '| P(X/Y/Z):', probe['probabilities'])


In [ ]:
# ★ Main 72 trials (resumes automatically after a restart).
main_rows, main_summary, main_folder = run_experiment(
    name='main_72', sampling='coverage', control='cyclic', modality='images')


In [ ]:
# OPTIONAL: uncomment only the needed controls. The same 24 scene sets are used.
# text_rows, text_summary, _ = run_experiment('text_only',control='fixed',modality='text')              # 24
# caption_rows, caption_summary, _ = run_experiment('captioned',control='fixed',modality='captioned')   # 24
# position_rows, position_summary, _ = run_experiment('position_order',control='position')            # 144
# relabel_rows, relabel_summary, _ = run_experiment('label_assignment',control='relabel')             # 144
# iid_rows, iid_summary, _ = run_experiment('iid_positive',sampling='iid',control='fixed')             # 24

## ★ Diagnostics · functions only until called

**Decisive ablation:** evaluate the whole ten-image set, remove its unique one-hot scene, then remove a non-decisive scene as a control. **Competence checks:** object recognition and decomposed Boolean gates.

In [ ]:
def drop_scene(scene, j):
    selected=[x for k,x in enumerate(scene['items']) if k!=j]
    return dict(scene, items=selected, scene_bits=[x['bits'] for x in selected],
                image_files=[x['filename'] for x in selected],
                consistent_candidates=compatible_rules(selected),
                identifiable=compatible_rules(selected)==[scene['hidden_rule']])

def run_ablation(n_sets=SETS_PER_RULE):
    mapping,display=layouts('fixed')[0]
    jobs=[]
    for rule in RULES:
        for i in range(n_sets):
            scene=sample_set(rule,i,'decisive_once')
            jobs.append((scene['group_id'],scene))
    cfg={'version':1,'model':MODEL_ID,'data_hash':DATA_HASH,'n_sets':n_sets,'seed':SEED}
    def infer(scene):
        bits=scene['scene_bits']; rule=scene['hidden_rule']
        decisive=next(i for i,b in enumerate(bits) if sum(b)==1)
        control=next(i for i,b in enumerate(bits) if sum(b)>1)
        minus_dec=drop_scene(scene,decisive); minus_ctl=drop_scene(scene,control)
        assert set(minus_dec['consistent_candidates'])==set(RULES)
        assert minus_ctl['consistent_candidates']==[rule]
        full=score_trial(scene,mapping,display)
        dec=score_trial(minus_dec,mapping,display)
        ctl=score_trial(minus_ctl,mapping,display)
        return dict(group_id=scene['group_id'],hidden_rule=rule,scene_bits=bits,
                    decisive_index=decisive+1, control_index=control+1,
                    full=full['probabilities'],minus_decisive=dec['probabilities'],
                    minus_control=ctl['probabilities'],
                    full_correct=full['correct'],
                    decisive_drop=full['true_probability']-dec['true_probability'],
                    control_drop=full['true_probability']-ctl['true_probability'])
    rows,folder=saved_runner('ablation',cfg,jobs,infer)
    summary={'n':len(rows),'full_accuracy':float(np.mean([r['full_correct'] for r in rows])),
             'mean_decisive_drop':float(np.mean([r['decisive_drop'] for r in rows])),
             'mean_control_drop':float(np.mean([r['control_drop'] for r in rows])),
             'mean_paired_effect':float(np.mean([r['decisive_drop']-r['control_drop'] for r in rows]))}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2))
    print(summary)
    return rows,summary

In [ ]:
def run_perception():
    """Eight held-out images; parse A/B/C. Note: only tests one variation per type."""
    jobs=[(str(r['filename']),r) for r in records if r['variation']==4]
    cfg={'version':1,'model':MODEL_ID,'data_hash':DATA_HASH,'modality':'images','variation':4}
    prompt="""Inspect this image. A = green cube, B = blue sphere, C = red cylinder.
Each means exact colour AND shape. Ignore distractors.
Reply with three binary values in format A=0, B=0, C=0 (replace as needed)."""
    def infer(r):
        reply=str(vlm.chat([str(IMAGE_DIR/r['filename'])],prompt,
                           max_new_tokens=60,temperature=0.0))
        found=[re.search(rf'\b{v}\s*[:=]\s*([01])',reply,re.I) for v in 'ABC']
        pred=[int(m.group(1)) if m else None for m in found]
        return dict(group_id=r['filename'],hidden_rule=None,filename=r['filename'],
                    ground_truth=r['bits'],prediction=pred,response=reply,
                    correct=pred==r['bits'])
    rows,folder=saved_runner('perception',cfg,jobs,infer)
    summary={'n':len(rows),'accuracy':float(np.mean([r['correct'] for r in rows]))}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2))
    print(summary)
    return rows,summary

def run_gates():
    """3 Boolean rules x 8 inputs; compare intermediate D and final Y."""
    indices={'R1':(0,1,2),'R2':(0,2,1),'R3':(1,2,0)}
    jobs=[(f'{rule}:{"".join(map(str,bits))}',(rule,bits))
          for rule in RULES for bits in product((0,1),repeat=3)]
    cfg={'version':1,'model':MODEL_ID,'task':'two_gate_text','rule_order':list(RULES)}
    def infer(pair):
        rule,bits=pair; i,j,k=indices[rule]; symbols='ABC'
        prompt=f"""A={bits[0]}, B={bits[1]}, C={bits[2]}.
D = {symbols[i]} AND {symbols[j]}
Y = D OR {symbols[k]}
Use Boolean gates. Reply exactly on two lines: D=<0/1> and Y=<0/1>."""
        response=str(vlm.chat([],prompt,max_new_tokens=50,temperature=0.0))
        def read(v):
            m=re.search(rf'\b{v}\s*[:=]\s*([01])',response,re.I)
            return int(m.group(1)) if m else None
        d,y=read('D'),read('Y'); d_true=bits[i]&bits[j]; y_true=int(satisfies(rule,bits))
        return dict(group_id=f'{rule}:{bits}',hidden_rule=rule,bits=list(bits),
                    predicted_d=d,predicted_y=y,expected_d=d_true,expected_y=y_true,
                    d_correct=d==d_true,y_correct=y==y_true,response=response)
    rows,folder=saved_runner('gates',cfg,jobs,infer)
    summary={'n':len(rows),'d_accuracy':float(np.mean([r['d_correct'] for r in rows])),
             'y_accuracy':float(np.mean([r['y_correct'] for r in rows])),
             'both_correct':float(np.mean([r['d_correct'] and r['y_correct'] for r in rows]))}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2))
    print(summary)
    return rows,summary

In [ ]:
# ★ Execute diagnostics selectively. All are resumable; nothing runs until uncommented.
# ablation_rows, ablation_summary = run_ablation()            # 24 sets × 3 calls = 72
# perception_rows, perception_summary = run_perception()      # 8 images
# gates_rows, gates_summary = run_gates()                      # 24 text-only


**Interpretation:** The experiments, underlying image data, prompts, seed and analysis functions match the Qwen3-VL minimal notebook; only the model-specific input/forward/generation adapter has changed. Candidate probabilities are **conditional on X/Y/Z** at the first generated token, not free-text answer likelihoods. Qwen3.5 thinking is explicitly disabled for this scoring protocol.

**Comparison cautions:** Because the old model used Roshan's wrapper and this model uses a native Qwen3.5 processor, image preprocessing and chat framing might not be byte-identical. The original 72-trial run always chose X; inspect the three-label sanity probabilities and the relabel/position controls before attributing a difference to reasoning. The ablation deliberately makes some nine-image cases ambiguous; report probability changes rather than accuracy after deletion. Results are separate from the Qwen3-VL run and no accuracies are pre-filled.

**Environment:** Install modern Transformers 5.x. This notebook has been syntax- and structural-tested but **not run through actual Qwen3.5 GPU inference**.
